# init-process-group-nccl — worked example 3: Selecting the backend by availability

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `init-process-group-nccl`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The only real difference between a GPU and CPU distributed setup is the backend string: `'nccl'` for CUDA GPUs, `'gloo'` for CPU. A small helper picks `'nccl'` when CUDA is available and falls back to `'gloo'` otherwise, so the same worker code runs on a laptop and a multi-GPU box.

## Worked solution

We select the backend dynamically, then init.

1. `pick_backend()` returns `'nccl'` if `t.cuda.is_available()` else `'gloo'`. NCCL requires real GPUs; gloo works on CPU.
2. The worker sets the rendezvous env vars as usual.
3. It calls `init_process_group(backend=pick_backend(), ...)` — identical to the fixed-backend version except the backend is computed.
4. After querying rank/world and doing its work, it destroys the group. On a CPU runtime this picks gloo and runs; on a GPU box the same code uses nccl. We print the chosen backend to make the selection visible.

In [ ]:
import os
import datetime
import torch as t
import torch.multiprocessing as mp
import torch.distributed as dist

def pick_backend():
    return 'nccl' if t.cuda.is_available() else 'gloo'

def worker(rank, world_size, port):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    backend = pick_backend()
    dist.init_process_group(
        backend=backend, rank=rank, world_size=world_size,
        timeout=datetime.timedelta(seconds=20),
    )
    print(f'[rank {rank}] backend={backend} world={dist.get_world_size()}')
    dist.destroy_process_group()

if __name__ == '__main__':
    print('chosen backend:', pick_backend())
    procs = [mp.Process(target=worker, args=(r, 2, 29512)) for r in range(2)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    print('exit codes:', [p.exitcode for p in procs])